
Geo Data Science with Python,
Prof. Susanna Werth, VT Geosciences

# Spatial Data: Preprocessing & Geostatistics


This notebook is accompanied by the lecture L05 presentation slides.

---


Content:
-------
- **A.** Getting Spatial Data: Importing Raster and Shapefiles
- **B.** Preprocessing Spatial Data & Geospatial Statistics


Preparation:
-------

On Colab, install the following tools:

* cartopy
* rasterio

Using the following code, download & unpack the following files to the folder you work in:

*   gt30w100n40_midEastUSA.tif
*   USA_adm1.dbf, .prj, .shp, .shx
*   USA_adm2.dbf, .prj, .shp, .shx


In [ ]:
import requests
import os

files = ["gt30w100n40_midEastUSA.tif", "USA_adm1.zip", "USA_adm2.zip"]

# Folder to save files
os.makedirs("dataLoaded", exist_ok=True)

base_url = "https://raw.githubusercontent.com/GeoPythonVT/geosf25_material/main/data/"

for fname in files:
    url = base_url + fname
    r = requests.get(url)
    r.raise_for_status()  # raise error if download failed
    with open(os.path.join("dataLoaded", fname), "wb") as f:
        f.write(r.content)


In [ ]:
import zipfile

zip_path = "./dataLoaded/USA_adm1.zip"   # path to your downloaded zip
extract_dir = "./dataLoaded/"

# Make sure the folder exists
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

In [ ]:
import zipfile

zip_path = "./dataLoaded/USA_adm2.zip"   # path to your downloaded zip
extract_dir = "./dataLoaded/"

# Make sure the folder exists
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

---
## Part A. Getting Spatial Data: Importing Raster and Shapefiles

Today, we work with the following datasets, which are available for download from the data folder on Github

**1. GTOPO30 DEM for VA, NC, and SC**

- Source: https://earthexplorer.usgs.gov and https://doi.org/10.5066/F7DF6PQS
- Geotiff file: gt30w100n40_SEUSA.tif


**2. Boundary Shapefiles for US states and counties**

- Source: https://gadm.org/
- Shape file: USA_adm1.shp (states), USA_adm2.shp (counties)

**3. GISTEMP data for VA, NC, and SC**

- ...


---
### 1. Read Geotif file with Rasterio

#### Documentation pages:
- https://github.com/rasterio/rasterio
- https://rasterio.readthedocs.io/en/stable/

#### Most important funcitons & methods:
- open(): opens raster file and returns an opened dataset object.
- read(): reads raster dataset by band’s index number (bands are indexed from 1, following the GDAL convention)

#### Most important dataset attributes read with the data, if available in the metadata of the file:
- transform
- crs
- nodata
- bounds

In [ ]:
import rasterio
import numpy as np

# Open GeoTIFF
file_path = "./dataLoaded/gt30w100n40_midEastUSA.tif"
with rasterio.open(file_path) as src:
    data = src.read(1)  # read first band (1-based indexing)
    transform = src.transform  # affine transform
    raster_crs = src.crs            # coordinate reference system
    nodata = src.nodata            # Get the nodata value

    # Compute coordinates of cell centers for coordinate vectors
    nrows, ncols = src.height, src.width  # Get arrays of row and column indices
    x_coords = transform.c + transform.a * (np.arange(ncols) + 0.5)
    y_coords = transform.f + transform.e * (np.arange(nrows) + 0.5)  # e is usually negative

    # Generate 2D coordinate grids from coordinate vectors
    xs, ys = np.meshgrid(x_coords, y_coords)

print(data.shape)
print(raster_crs)
print(type(data))
print("EPSG:4326 stands for WGS84")

Let's have a brief look into the data with cartopy.

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np

masked_data = np.ma.masked_equal(data, nodata)  # use numpy to generate a masked array
#masked_data = np.where(data == nodata, np.nan, data)  # alternatively set nodata values to NaN

# Create a figure with Cartopy projection
fig, ax = plt.subplots(figsize=(6,4), subplot_kw={'projection': ccrs.PlateCarree()})

# plot as image, no coordinates required:
#extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]
#im = ax.imshow(masked_data, extent=extent, origin='upper', transform=ccrs.PlateCarree(), cmap='viridis')

# plot as meshgrid with converted coordinates:
im = plt.pcolormesh(xs, ys, masked_data, shading='auto', cmap='viridis')

# Add gridlines & labels
gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', linestyle='--')
gl.top_labels = False     # Disable top labels
gl.right_labels = False   # Disable right labels
gl.bottom_labels = True
gl.left_labels = True
#ax.coastlines(resolution='110m')
plt.colorbar(im, ax=ax, label='Raster Value')
plt.title('GeoTIFF plotted with Cartopy')
plt.show()


We will continue to work with the datasets in Part B.

### Read Shapefiles with GeoPandas

GeoPandas extends the data types used by pandas to enable spatial operations on geometric types. Geometric operations are performed by shapely, plotting using matplotlib, and file access using pyogrio.

#### Documentation pages:
- https://geopandas.org/en/

#### Data Objects
- GeoDataFrame: tabular data structure that contains a GeoSeries, subclass of Pandas.DataFrame
- GeoSeries: vector containing a set of shapes, subclass of Pandas.Series

#### Most important functions & methods of GeoDataFrame:
- read_file(): reads vector-based files into a GeoPandas data object (GeoSeries or GeoDataFrame)
- to_crs(): change coordinate reference systems
- plot(): plot GeoSeries
- ... (there are many more, we will cover some more later)

#### Most important attributes :
- crs
- columns
- head()
- area: shape area
- bounds: bounding box coordinates
- geom_type: type of geometry
- centroid: returns GeoSeries of centroids
- ... (there are many more, we will cover some more later)


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

# Load shapefile
shapefile_path = "./dataLoaded/USA_adm1.shp"  # State boundaries
gdf1 = gpd.read_file(shapefile_path)

# Check the data
print(' ')
print(gdf1.crs)  # Coordinate Reference System
print(' ')
print(gdf1.columns)  # Attribute columns
print(' ')
gdf1.head()

The shapefile contains multi-polygons, with smaller island polygons for some states. The next part of the code will search for the largest polygon, and just keep that.

In [ ]:
# 1. Keep only Polygons / MultiPolygons
gdf_polygons = gdf1[gdf1.geom_type.isin(['Polygon', 'MultiPolygon'])].copy()

# 2. Reproject to a metric CRS for accurate area computation
# Example: US Albers Equal Area (EPSG:5070)
gdf_projected = gdf_polygons.to_crs(epsg=5070)

# 3. Explode MultiPolygons
gdf_exploded = gdf_projected.explode(index_parts=False).reset_index(drop=True)

# 4. Keep original indices to group by
gdf_projected['orig_index'] = gdf_projected.index
gdf_exploded = gdf_projected.explode(index_parts=False).reset_index(drop=True)

# 5. Compute area in projected CRS
gdf_exploded['area'] = gdf_exploded.geometry.area

# 6. Select the largest polygon for each original feature
largest_polys = gdf_exploded.loc[gdf_exploded.groupby('orig_index')['area'].idxmax()]
largest_polys = largest_polys.drop(columns=['area', 'orig_index'])

# 7. Reproject back to geographic CRS for plotting
gdf_largest = largest_polys.to_crs(epsg=4326)

# 8. Plot with GeoPandas plot attribute
gdf_largest.plot(figsize=(4, 2))

In [ ]:
# 9. Plot with Matplotlib or Cartopy
fig, ax = plt.subplots(figsize=(4,2))
gdf_largest.plot(ax=ax, facecolor='orange', edgecolor='black', alpha=0.6)
ax.set_title("USA States: Largest of the MultiPolygon")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.show()


Now let's get only the row and then read out the polygon information for North Carolina.

In [ ]:
NC_gdf = gdf_largest[gdf_largest['NAME_1'] == 'North Carolina']
NC_gdf.plot(facecolor='orange', edgecolor='black', alpha=0.6)
NC_gdf

In [ ]:
NC_poly = NC_gdf.geometry
NC_poly

This polygon is stored in an object type `geometry` which is provided by the package shapely, running in the background. We will continue to work with this in Part C.

---

## Exercise A

1. Import & inspect the shapefile for US counties (in the file `USA_adm/USA_adm2.shp`).
2. Find and select the polygon for Montgomery County.

#### Extra Credit
3. Loop throught the states shapefile `(gdf1 / gdf_largest)` in a way, that the following states are plotted in a map:
- Arizona
- Alabama
- Ohio
- Virginia


---
## Part B. Preprocessing Spatial Data & Geostatistics

- Cropping & Masking
- Zonal Statistics
- Buffering
- Resampling & Interpolation
- Filtering / Smoothing
- Variogramm analysis & Kriging

### 1. Cropping & Masking

Cropping and masking are useful operations, allowing to analyze only specific regions in the raster array. We already have a masked array, that focuses the dataset on land area and masks out coastal areas. Now, we want to mask out the topographical raster data within the state of NC. For that, we will create a gridded mask using the NC polygon.

Masking & cropping operations are primarily relevant to rastered/gridded data, although they can also be applied to a shapefile. For example, we have just preselected the state of NC by identifying and isolating the relevant polygon from a national dataset.


#### Cropping, if necessary

If you have a big dataset, cropping the dataset first to a smaller bounding box can speed up the mask creation later on. For the size of our dataset today, this is not necessary. But if you get along this problem in the future, here is an example.

In [ ]:
# Getting min/max coordinates from the NC polygon
NCbnds = NC_poly.bounds
NCbnds


In [ ]:
minx, maxx = NCbnds.minx.iloc[0], NCbnds.maxx.iloc[0]
miny, maxy = NCbnds.miny.iloc[0], NCbnds.maxy.iloc[0]


In [ ]:

# Find boolean mask of cells within bounds
mask = (xs >= minx) & (xs <= maxx) & (ys >= miny) & (ys <= maxy)

# finding the correct index in the DEM array
rows, cols = np.where(mask)
row_start, row_end = rows.min(), rows.max() + 1
col_start, col_end = cols.min(), cols.max() + 1

# Slice the arrays down to the NC bounds
masked_data_crop = masked_data[row_start:row_end, col_start:col_end]
xs_crop = xs[row_start:row_end, col_start:col_end]
ys_crop = ys[row_start:row_end, col_start:col_end]


Now you could continue working with the cropped dataset and coordinates, to make the next steps faster.

In [ ]:
fig, ax = plt.subplots(figsize=(3,2))
plt.pcolormesh(xs_crop, ys_crop, masked_data_crop, shading='auto', cmap='viridis')
plt.show()

#### Masking

For this task, the `shapely` module comes in very handy. It provides objects for points, lines and polygons, and also GeoPandas relies heavily on those object types. If you are interested in a comprehensive overview of shapely, go through the basic tutorial `B09_shapely.ipynb`. The documentation pages of shapely are: https://shapely.readthedocs.io/en/stable.

Here we will use some of the most important elements of shapely.

One useful shapely functions is `contains()` and `contains_xy()`, which can be used to check if a polygon contains other elements. Here, we will use the `contains_xy()` function to create a raster mask for the state polygon we retrieved earlier.


In [ ]:
from shapely.geometry import Polygon
from shapely import contains_xy  # function in Shapely 2.0+

# Create a boolean mask: True for cells inside the polygon
mask = contains_xy(NC_poly, xs, ys)  # True for cells inside the polygon

# Combine masks: True means the cell will be masked
combined_mask = (~mask) | (data == nodata)

# Apply to masked array
masked_data_roi = np.ma.array(data, mask=combined_mask) # mask=True means hide


In [ ]:
# plot the results
fig, ax = plt.subplots(figsize=(3,2))
plt.pcolormesh(xs, ys, masked_data_roi, shading='auto', cmap='viridis')
plt.show()

### 2. Zonal Statistics

The newly created mask will now enable us to use it for estimating statistics from the raster dataset within the region for which the mask was created, specifically the state of NC.

**Task**: Let's estimate the min, max, mean, and median height for NC and Montgomery County.

In [ ]:
# Estimate the min, max, mean and median height for NC
# Does it make a difference if you use the np.nan... functions or not?






### 3. Buffering via Shapely


In [ ]:
from shapely.geometry import Point, LineString
BBpoint = Point(-80.413940, 37.229572) # create a shapely point at location of Blacksburg
CHpoint = Point(-80.843124, 35.227085) # create a shapely point at location of Charlotte
CHBBline = LineString((BBpoint, CHpoint)) # create a line connecting Blacksburg and Charlotte (Earth curvature neglected)
CHBBline

In [ ]:
buffered_poly1 = CHpoint.buffer(0.5)  # distance in the same units as your coordinates
# Positive distance: expands polygon
# Negative distance: contracts polygon
buffered_poly1

In [ ]:
buffered_poly2 = CHBBline.buffer(0.1)  # distance in the same units as your coordinates
buffered_poly2

In [ ]:
# Get coordinates for buffer polygons
exterior_coords1 = list(buffered_poly1.exterior.coords)
yCHb = [ e[1] for e in exterior_coords1 ]
xCHb = [ e[0] for e in exterior_coords1 ]
exterior_coords2 = list(buffered_poly2.exterior.coords)
yCHBBline = [ e[1] for e in exterior_coords2 ]
xCHBBline = [ e[0] for e in exterior_coords2 ]


In [ ]:
# Plot buffer in a map
fig, ax = plt.subplots(figsize=(3,2))
plt.pcolormesh(xs, ys, masked_data_roi, shading='auto', cmap='viridis')
plt.plot(xCHb,yCHb, 'r')
plt.plot(xCHBBline,yCHBBline, 'r')
plt.show()

---

## Exercise B.1

1. Create a masked array, that only contains data for the state of Virginia. Estimate min, max, and mean elevation in Virginia.
2. A buffer along a line between two cities can represent a transport zone between them. Create such a buffer from Blacksburg to Harrisonburg (Latitude: 38.4495700°, Longitude: -78.8689200°) as well as a buffer surrounding the town of Blacksburg (similar to above). Use 0.1 degree (~10km) for the transportation zone and 0.5 degree (~50km) for the town. Then, use both buffers to mask out topographic data and estimate the min, max, and mean elevation values in the buffer zones. 
3. Create a new buffer polygon with a buffer of 0.5 extending outside Virginia. Plot the polygon.


#### Extra credit:

4. Investigate the shapely functions contains() and within(). What is their difference? Can both functions achieve the same?
5. Read the code on cropping. Briefly explain the concept. Then generate a cropped dataset within the bounding box of the state of VA.


### 4. Resampling & Interpolation

First, let's create a cropped dataset, then from that a downsampled version.

In [ ]:
# plot as meshgrid with converted coordinates:
lmin = 250
lmax = 500
bmin = 0
bmax = 250

# cropped dataset (mid north section in original map)
xsN = xs[lmin:lmax,bmin:bmax]
ysN = ys[lmin:lmax,bmin:bmax]
dataN = data[lmin:lmax,bmin:bmax]

# downsampled dataset (keep every 3rd point)
dd = 3
xsN_dwnsmpld = xsN[::dd,::dd]
ysN_dwnsmpld = ysN[::dd,::dd]
dataN_dwnsmpld = dataN[::dd,::dd]


Next, we want to interpolate the data back onto the orignial resolution using `griddata` https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.griddata.html

This way is not fast because it cannot take advantage of structured (predictably-spaced) data. However, it generally works and is pretty straight-forward.

    Z = scipy.interpolate.griddata(pts, z, xy, method='cubic', fill_value=0)

where `pts` is Nx2 and contains the coordinates for the input data, `z` are the values at `pts`, `xy` are the coordinates where you want to interpolate to, `Z` are the values of the function `z` at `xy` locations. `method` can be 'linear', 'nearest', 'cubic', and `fill_value` fills in outside of the points.

The function is actually made for interpolation of unstructured (unregularly sampled) datapoints to a regular grid.

In [ ]:
import scipy

pts = np.vstack((xsN_dwnsmpld.flat,ysN_dwnsmpld.flat)).T  # combine x and y to get Nx2 array

# Now interpolate to new values
xy = np.vstack((xsN.flat, ysN.flat)).T
Zgrid = scipy.interpolate.griddata(pts, dataN_dwnsmpld.flat, xy, method='linear', fill_value=0)

# reconstitute the output to structured array so we can plot it with pcolormesh
Zgrid.shape = xsN.shape

In [ ]:
fig, axes = plt.subplots(1, 3, sharex=True, sharey=True)
axes[0].pcolormesh(xsN, ysN, dataN, cmap='viridis') # Original
axes[1].pcolormesh(xsN_dwnsmpld, ysN_dwnsmpld, dataN_dwnsmpld, cmap='viridis')
#axes[1].scatter(xsN_dwnsmpld, ysN_dwnsmpld, c=dataN_dwnsmpld, s=10, cmap='viridis') # downsampled
axes[2].pcolormesh(xsN, ysN, Zgrid, cmap='viridis') # Interpolated to original grid
axes[0].set_title('Original')
axes[1].set_title('Downsampled')
axes[2].set_title('Interpolated')
plt.show()

#### Note: Interpolation on a map

The curvature of the earth needs to be considered when coordinates that are in longitude/latitude. When latitude and longitude are changing uniformly, it is fine (such as when you are moving along a single latitude or single longitude), but when you are changing between multiple, the values of each decimal degree changes in space. This needs to be accounted for by first projecting your coordinates. This effect is smaller closer to the equator, but can be large closer to the poles.


#### Resampling

To resample, i.e., evaluate any point from a regular grid, the `RectBivariateSpline` function is useful.

In [ ]:
pts = np.array([[BBpoint.y, BBpoint.x], [CHpoint.y, CHpoint.x]])  # points to interpolate to

In [ ]:
if y_coords[0] > y_coords[-1]:
    y_coords_tmp = y_coords[::-1]
    data_tmp = np.flipud(masked_data)

In [ ]:
f = scipy.interpolate.RectBivariateSpline(y_coords_tmp, x_coords, data_tmp)
zinterp = f.ev(pts[:,0], pts[:,1])
zinterp

#### Delauney Triangulation in Detail (optional)

Internally, the linear and cubic interpolation with the methods above rely on Delaunay triangulation. Let's look into details fo that via an syntetic example of a Delauney triangulation:

In [ ]:
# Create triangulation
x = np.random.rand(10)
y = np.random.rand(10)
z = np.sin((x**2 + y**2)*5.0)
pts = np.vstack((x,y)).T

fig, axes = plt.subplots(1, 2, figsize=(4,2), sharex=True, sharey=True)
axes[0].scatter(x, y, c=z, s=100, cmap='viridis')

tri = scipy.spatial.Delaunay(pts)

# Plot triangles (filled with transparency)
for indices in tri.simplices:  # use .simplices instead of .vertices
    axes[1].fill(pts[indices, 0], pts[indices, 1], edgecolor='none', alpha=0.3)

# Plot original points
axes[1].plot(pts[:, 0], pts[:, 1], '.k')

# Plot convex hull
for indices in tri.convex_hull:
    axes[1].plot(pts[indices, 0], pts[indices, 1], 'r-')  # red lines

plt.show()

Now, let's continue with the real topography data. This time we select random points from the cropped original dataset:

In [ ]:
# select random points from the cropped original dataset
fd = np.unique(np.sort(np.round(np.random.rand(10000)*250*250))).astype(int)
xRnd = xsN.flatten()[fd]
yRnd = ysN.flatten()[fd]
dataRnd = dataN.flatten()[fd]

# rename for later convenience
pts = np.vstack((xRnd,yRnd)).T

fig, ax = plt.subplots(figsize=(2,3))
plt.scatter(xRnd, yRnd, c=dataRnd, s=1, cmap='viridis')
plt.show()

To use triangulation for interpolation, we will functions in the `scipy.interpolate` subpackage. Most of these functions use Delaunay interpolation under the hood. For example, you can pass an existing Triangulation to `scipy.interpolate.LinearNDInterpolator`. However, you can also just pass it the points you want, and it will do the triangulation for you.

In [ ]:

tri = scipy.spatial.Delaunay(pts)

# create an iterpolation object, f. We need to supply the data values on the specified xy points.
f = scipy.interpolate.LinearNDInterpolator(tri, dataRnd)
Z = f(xsN, ysN)  # this is the interpolation step
Z = np.ma.masked_where(np.isnan(Z), Z)

fig, axes = plt.subplots(1, 3, sharex=True, sharey=True)
axes[0].pcolormesh(xsN, ysN, dataN, cmap='viridis') # Original
axes[1].scatter(xRnd, yRnd, c=dataRnd, s=1, cmap='viridis')
axes[2].pcolormesh(xsN, ysN, Z, cmap='viridis') # Interpolated to original grid
axes[0].set_title('Original')
axes[1].set_title('Random Selection')
axes[2].set_title('Interpolated')
plt.show()


### 5. Filtering or Smoothing

Let's first add some noise to the data, then try to smooth it out.

In [ ]:
noRow, noCols = masked_data.shape
noisy_data = masked_data + (np.random.rand(noRow, noCols)-0.5)*300  # pretty strong noise (+/-150 m)

Let's demonstrate how filtering works via a convolusion of a filter matrix over the data matrix.

In [ ]:
from scipy.ndimage import convolve

# simple averaging using a 6x6 smoothing kernel
kernel = np.array([[1, 1, 1, 1, 1, 1],
                   [1, 1, 1, 1, 1, 1],
                   [1, 1, 1, 1, 1, 1],
                   [1, 1, 1, 1, 1, 1],
                   [1, 1, 1, 1, 1, 1],
                   [1, 1, 1, 1, 1, 1]]) / 36
smoothed_conv = convolve(masked_data.filled(0), kernel)
smoothed_conv = np.ma.array(smoothed_conv, mask=masked_data.mask)

fig, axes = plt.subplots(1, 2, figsize=(6, 3))  # 1 row, 2 columns
im1=axes[0].pcolormesh(xs, ys, masked_data, shading='auto', cmap='viridis')
im2=axes[1].pcolormesh(xs, ys, smoothed_conv, shading='auto', cmap='viridis')
plt.show()


In real data science life, we can use ready made filters, useful for smoothing and feature extraction.

In [ ]:
from scipy.ndimage import gaussian_filter

# Apply Gaussian filter
# sigma controls the smoothing (in number of pixels)
sigma = 2
smoothed_data = gaussian_filter(noisy_data.filled(0), sigma=sigma)
# the method .filled() sets NaN in the masked array to a value, here 0

# Optional: reapply nodata mask
smoothed_data = np.ma.array(smoothed_data, mask=noisy_data.mask)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(6, 3))  # 1 row, 2 columns
im1=axes[0].pcolormesh(xs, ys, masked_data, shading='auto', cmap='viridis')
im2=axes[1].pcolormesh(xs, ys, noisy_data, shading='auto', cmap='viridis')
im3=axes[2].pcolormesh(xs, ys, smoothed_data, shading='auto', cmap='viridis')
plt.colorbar(im1, ax=axes[0], fraction=0.04, pad=0.14,orientation='horizontal') # fraction controls colorbar thickness
plt.colorbar(im2, ax=axes[1], fraction=0.04, pad=0.14,orientation='horizontal') # fraction controls colorbar thickness
plt.colorbar(im3, ax=axes[2], fraction=0.04, pad=0.14,orientation='horizontal') # pad controls space between axes and colorbar
axes[0].set_title('Original')
axes[1].set_title('Noisy')
axes[2].set_title('Smoothed')
plt.show()

Other useful filter functions that scipy provides:

- **Gaussian Filter** (used above): Smooths using a weighted kernel; stronger smoothing at higher sigma. Suitable for general-purpose smoothing, such as noise reduction.
- **Mean Filter / Uniform Filter**: Replaces each cell with the average of neighboring cells. Smooths the raster but keeps edges less blurred than Gaussian.
- **Median Filter**: Replaces each cell with the median of neighboring cells. Good for removing salt-and-pepper noise.
- **Laplacian / Edge Detection Filter**: Highlights edges and abrupt changes. Often used for feature detection.
- **Convolution with a Custom Kernel**: you can always create your own filter and apply that.

### 6. Variogramm analysis
This is useful to get insight into how similar values are with distance. Variograms also server as input to Kriging approches, which are sophisticated interpolation techniques, that consider variability in the data and also allow estimation of interpolation uncertaintz.

In [ ]:
# Use only portion of the dataset
lmin = 250
lmax = 500
bmin = 0
bmax = 250

# cropped dataset (mid north section in original map)
xsN = xs[lmin:lmax,bmin:bmax]
ysN = ys[lmin:lmax,bmin:bmax]
dataN = data[lmin:lmax,bmin:bmax]

# select random points from the cropped original dataset
fd = np.unique(np.sort(np.round( np.random.rand(20000)*dataN.size-1 ))).astype(int)
xRnd = xsN.flatten()[fd]
yRnd = ysN.flatten()[fd]
dataRnd = dataN.flatten()[fd] 

# Mask nodata
mask = ~np.isnan(dataRnd)
x, y, z = xRnd[mask], yRnd[mask], dataRnd[mask]
coords = np.column_stack([x, y])


In [ ]:
# Compute pairwise distances and squared differences

from scipy.spatial.distance import pdist

# Pairwise distances between all points
dists = pdist(coords)  # returns array of distances between all pairs

# Pairwise squared differences / 2 (semivariance)
diffs = pdist(z.reshape(-1,1), metric='sqeuclidean') / 2.0

In [ ]:
# Bin distances into lag intervals

# Define lag bins
max_lag = np.max(dists) / 2  # e.g., half of max distance
n_lags = 15
bins = np.linspace(0, max_lag, n_lags + 1)

# Assign each distance to a bin
bin_indices = np.digitize(dists, bins)

# Compute average semivariance per bin
semivariance = np.array([diffs[bin_indices == i].mean() for i in range(1, len(bins))])

# Compute lag centers for plotting
lag_centers = (bins[:-1] + bins[1:]) / 2

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
plt.plot(lag_centers, semivariance, 'o-', label='Experimental Variogram')
plt.xlabel("Lag distance")
plt.ylabel("Semivariance")
plt.title("Manual Variogram")
plt.legend()
plt.show()

**Task**: 4. Discuss the variogram that we retrieved, how do you interpret it? 


---

## Exercise B.2

1. Interpolate the elevation of Derring Hall from the elevation dataset. Collect the coordinates from an online map.
2. Choose a filter method, then filter the entire dataset. Plot two cross-sections, one in latitude and one in longitude direction (choose any you find interesting), from the filtered and unfiltered datasets. Compare and discuss the outcomes.
3. Use shapely documentation pages or ask an AI how to apply the Laplacian filter to your dataset. Compare and discuss the output with a simple averaging (mean) filter.

#### Extra credit:

4. Experiment with the convolution filter; you can also enter negative values. What is the effect of the size of the convolution matrix (a smaller being 3x3 versus a larger, e.g., 10x10), e.g. for the simple averaging filter? And how do you think the convolution kernel would look like (approximately) for a Laplacian filter?
5. Estimate a variogram after adding noise for the dataset. How does it change?
